# 03 â€” Train tabular models (RF, XGBoost, PTT-LR)

Loads `features.parquet`, derives the 3-class AHA label from `sbp`/`dbp`, splits subjects into train + held-out test, and trains three tabular models on the engineered features:

- **Random Forest** â€” strong all-around tabular baseline.
- **XGBoost** â€” typically the best engineered-feature model on this kind of data.
- **PTT-only logistic regression** â€” single-feature baseline (just `ptt_ms`), matching the writeup's "PTT regression" reference point.

All three reuse the same subject-level GroupKFold CV splits (`splits.json`), and the CNN in nb 05 reads the same file so test-set comparisons are apples-to-apples.

Outputs:
- `models/rf_3class.joblib`, `models/xgb_3class.joblib`, `models/ptt_lr.joblib`
- `data/processed/{rf,xgb,ptt_lr}_metrics.json` â€” `MultiClassMetrics`
- `data/processed/{rf,xgb}_importance.csv` â€” permutation importance

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold

from bme_ml.paths import setup_paths
from bme_ml.labels import add_multiclass_column, MULTICLASS_NAMES
from bme_ml.splits import make_splits, select_rows, add_subject_id
from bme_ml.models import (
    build_rf, tune_rf,
    build_xgb, tune_xgb,
    build_ptt_logreg,
    Xy, Xy_ptt_only,
)
from bme_ml.features import FEATURE_NAMES
from bme_ml.evaluation import evaluate_multiclass, feature_importance

LABEL_COL = 'label_3class'
paths = setup_paths()

In [2]:
features = pd.read_parquet(paths.features_parquet)
features = add_subject_id(features)
features = add_multiclass_column(features, col=LABEL_COL)
print('rows:', len(features), 'subjects:', features['subject_id'].nunique())
print('class distribution:')
print(features[LABEL_COL].value_counts().rename(index=dict(enumerate(MULTICLASS_NAMES))))

splits = make_splits(features, label_col=LABEL_COL)
splits.to_json(paths.splits_json)
print('test subjects:', len(splits.test_subjects), 'cv folds:', len(splits.cv_folds))
assert set(features[LABEL_COL].unique()) <= {0, 1, 2}, 'unexpected class label'

rows: 128144 subjects: 11987
class distribution:
label_3class
Hypertensive    60802
Normal          47362
Elevated        19980
Name: count, dtype: int64


test subjects: 2398 cv folds: 5


In [3]:
train_subjects = sorted({s for fold in splits.cv_folds for s in fold['train']} |
                        {s for fold in splits.cv_folds for s in fold['val']})
train_df = select_rows(features, train_subjects)
test_df = select_rows(features, splits.test_subjects)

X_train, y_train = Xy(train_df, label_col=LABEL_COL)
X_test, y_test = Xy(test_df, label_col=LABEL_COL)
print('full features:', X_train.shape, '->', X_test.shape)

X_train_ptt, y_train_ptt = Xy_ptt_only(train_df, label_col=LABEL_COL)
X_test_ptt, y_test_ptt = Xy_ptt_only(test_df, label_col=LABEL_COL)
print('PTT only    :', X_train_ptt.shape, '->', X_test_ptt.shape)

# CV splits over the full-features train pool (used for RF and XGBoost).
cols = [c for c in FEATURE_NAMES if c in train_df.columns]
train_clean = train_df.dropna(subset=cols + [LABEL_COL]).reset_index(drop=True)
groups = train_clean['subject_id'].to_numpy()
gkf = GroupKFold(n_splits=5)
cv_splits = list(gkf.split(train_clean, groups=groups))

full features: (27478, 20) -> (6968, 20)
PTT only    : (56512, 1) -> (14151, 1)


## Random Forest

In [4]:
rf_grid = tune_rf(X_train, y_train, cv_splits)
rf = rf_grid.best_estimator_
print('RF best params :', rf_grid.best_params_)
print('RF CV f1_macro :', rf_grid.best_score_)

rf_pred = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)
rf_metrics = evaluate_multiclass(y_test, rf_pred, rf_proba)
print(rf_metrics)

joblib.dump(rf, paths.models / 'rf_3class.joblib')
(paths.processed / 'rf_metrics.json').write_text(json.dumps(rf_metrics.__dict__, indent=2))
rf_imp = feature_importance(rf, X_test, y_test, feature_names=cols)
rf_imp.to_csv(paths.processed / 'rf_importance.csv', index=False)
rf_imp.head(8)

Fitting 5 folds for each of 36 candidates, totalling 180 fits


RF best params : {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 5, 'n_estimators': 500}
RF CV f1_macro : 0.5854979985214713


MultiClassMetrics(accuracy=0.6591561423650976, f1_macro=0.5825143580606672, confusion=[[1732, 235, 643], [335, 318, 430], [498, 234, 2543]], hypertensive_auroc=0.8194873321303148, hypertensive_pr_auc=0.7898855668936557, hypertensive_recall=0.7764885496183206, hypertensive_false_negative_rate=0.2235114503816794)


,feature,importance_mean,importance_std
0,hr_bpm,0.037414,0.003184
1,ecg_qrs_ms,0.025574,0.001874
2,ppg_pw50_ms,0.024770,0.002363
3,lasi_ms,0.023995,0.001760
4,ppg_pw25_ms,0.020020,0.001902
5,ppg_rise_ms,0.018441,0.002602
6,ppg_decay_ms,0.017738,0.001778
7,s1_area,0.012184,0.001898


## XGBoost

In [5]:
xgb_grid = tune_xgb(X_train, y_train, cv_splits)
xgb = xgb_grid.best_estimator_
print('XGB best params :', xgb_grid.best_params_)
print('XGB CV f1_macro :', xgb_grid.best_score_)
assert xgb.classes_.tolist() == [0, 1, 2], f'unexpected XGB classes_: {xgb.classes_.tolist()}'

xgb_pred = xgb.predict(X_test)
xgb_proba = xgb.predict_proba(X_test)
xgb_metrics = evaluate_multiclass(y_test, xgb_pred, xgb_proba)
print(xgb_metrics)

joblib.dump(xgb, paths.models / 'xgb_3class.joblib')
(paths.processed / 'xgb_metrics.json').write_text(json.dumps(xgb_metrics.__dict__, indent=2))
xgb_imp = feature_importance(xgb, X_test, y_test, feature_names=cols)
xgb_imp.to_csv(paths.processed / 'xgb_importance.csv', index=False)
xgb_imp.head(8)

Fitting 5 folds for each of 48 candidates, totalling 240 fits


XGB best params : {'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 8, 'n_estimators': 500, 'subsample': 0.8}
XGB CV f1_macro : 0.5794567710052283
MultiClassMetrics(accuracy=0.6849885189437428, f1_macro=0.5825871826487283, confusion=[[1843, 150, 617], [412, 227, 444], [435, 137, 2703]], hypertensive_auroc=0.8504461711139085, hypertensive_pr_auc=0.8231961625129817, hypertensive_recall=0.8253435114503817, hypertensive_false_negative_rate=0.17465648854961835)


,feature,importance_mean,importance_std
0,ppg_pw50_ms,0.077138,0.003303
1,ecg_qrs_ms,0.039064,0.001584
2,hr_bpm,0.033654,0.003055
3,ppg_rise_ms,0.032348,0.002602
4,lasi_ms,0.029908,0.002721
5,ppg_decay_ms,0.029693,0.003109
6,ppg_amp_ratio,0.024455,0.002154
7,rr_sd_ms,0.022991,0.002377


## PTT-only logistic regression

Single-feature baseline using only `ptt_ms`. Directly mirrors the "PTT regression" baseline cited in the writeup template â€” if the engineered features contribute beyond pulse transit time alone, RF and XGBoost should clear this baseline by a margin.

In [6]:
ptt_lr = build_ptt_logreg()
ptt_lr.fit(X_train_ptt, y_train_ptt)
ptt_pred = ptt_lr.predict(X_test_ptt)
ptt_proba = ptt_lr.predict_proba(X_test_ptt)
# Pad probabilities to (n_samples, 3) if a class is absent from training
# (rare but possible on the smoke-test dataset).
if ptt_proba.shape[1] != 3:
    full = np.zeros((ptt_proba.shape[0], 3), dtype=ptt_proba.dtype)
    for i, c in enumerate(ptt_lr.classes_):
        full[:, int(c)] = ptt_proba[:, i]
    ptt_proba = full

ptt_metrics = evaluate_multiclass(y_test_ptt, ptt_pred, ptt_proba)
print(ptt_metrics)

joblib.dump(ptt_lr, paths.models / 'ptt_lr.joblib')
(paths.processed / 'ptt_lr_metrics.json').write_text(json.dumps(ptt_metrics.__dict__, indent=2))

MultiClassMetrics(accuracy=0.2579323016041269, f1_macro=0.23571684082679947, confusion=[[266, 3872, 1649], [99, 1540, 457], [193, 4231, 1844]], hypertensive_auroc=0.4996430121412706, hypertensive_pr_auc=0.473003748697507, hypertensive_recall=0.29419272495213783, hypertensive_false_negative_rate=0.7058072750478621)


423

In [7]:
from bme_ml.evaluation import compare_multiclass
compare_multiclass([
    ('Random Forest', rf_metrics),
    ('XGBoost', xgb_metrics),
    ('PTT-only LR', ptt_metrics),
])

,model,accuracy,f1_macro,hypertensive_auroc,hypertensive_recall,false_negative_rate
0,Random Forest,0.659156,0.582514,0.819487,0.776489,0.223511
1,XGBoost,0.684989,0.582587,0.850446,0.825344,0.174656
2,PTT-only LR,0.257932,0.235717,0.499643,0.294193,0.705807
